# Spark → Kusto — write via `KustoStreaming` (Custom Environment)

This notebook assumes a Fabric **Custom Environment** is already attached that ships `kusto-commons-io-shim-1.0.0.jar` and sets `userClassPathFirst=true` on driver + executor.

Setup instructions: see [`../environment/README.md`](../environment/README.md).
Background on *why* the shim is needed: see [`../README.md`](../README.md).

**Before running:**
1. Attach the `kusto-streaming-env` (or however you named it) via the **Environment** dropdown in the top-right of this notebook.
2. Restart the session so the JAR loads.
3. Confirm the target Kusto table has streaming ingestion enabled (KQL below).


## Prereq KQL — run once in your Eventhouse query set

```kusto
// Enable streaming ingestion on the target table
.alter table FreezerTelemetry policy streamingingestion enable

// Verify
.show table FreezerTelemetry policy streamingingestion
```


In [ ]:
// === Edit these to match your Eventhouse ===
val kustoUri  = "https://<your-cluster>.kusto.fabric.microsoft.com"
val database  = "<database>"
val tableName = "FreezerTelemetry"

val accessToken = mssparkutils.credentials.getToken(kustoUri)

println(s"Cluster:  $kustoUri")
println(s"Database: $database")
println(s"Table:    $tableName")

## Verify the shim is on the classpath

If either of these throws, the Environment isn't attached or hasn't fully loaded yet.

In [ ]:
// 1) Shim's IOUtils is loadable (proves the shim JAR is on the classpath)
Class.forName("kusto_connector_shaded.org.apache.commons.io.IOUtils")

// 2) Exactly one shim JAR is on the user classpath
spark.sparkContext.listJars()
  .filter(j => j.contains("kusto") || j.contains("commons-io-shim"))
  .foreach(println)

## Read from Kusto (sanity check)

In [ ]:
val df = spark.read
  .format("com.microsoft.kusto.spark.datasource")
  .option("accessToken",   accessToken)
  .option("kustoCluster",  kustoUri)
  .option("kustoDatabase", database)
  .option("kustoQuery",    s"['$tableName'] | take 10")
  .load()

df.show()
println(s"Read ${df.count()} rows")

## Write to Kusto with `writeMode=KustoStreaming`

This is the path that was broken before the shim — `NoClassDefFoundError` on `IOUtils`. With the shim on the classpath, it works as Microsoft intended.

In [ ]:
df.write
  .format("com.microsoft.kusto.spark.datasource")
  .option("kustoCluster",  kustoUri)
  .option("kustoDatabase", database)
  .option("kustoTable",    tableName)
  .option("writeMode",     "KustoStreaming")
  .option("accessToken",   accessToken)
  .mode("append")
  .save()

println("✅ Write completed")

## Verify in Kusto

```kusto
FreezerTelemetry
| where ingestion_time() > ago(2m)
| count
```

Streaming-ingested rows usually appear within 1–5 seconds.